## Black Scholes European option pricing

Begin by importing the necessary modules for the code

In [25]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm
from datetime import datetime
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

Then we can begin by defining the black scholes model, recall the model follows:

$$C = S_0 \Phi(d_1)-Ke^{-rT}\Phi (d_2)$$

$$d_1 = \frac{\ln\left(\frac{S_0}{K}\right) + \left(r+\frac{\sigma^2}{2}\right)T}{\sigma \sqrt{T}}$$

$$d_2 = d_1 - (\sigma  \sqrt{T})$$

Where:
- $C$: Call option price
- $S$: Spot price of an asset
- $K$: Strike Price
- $\sigma$: Volatility
- $r$: risk-free rate
- $T$: Time to maturity

Now to implement this into code:

In [26]:
def Black_Scholes_Call(S0, K, sigma, r, t):
    d1 = (np.log(S0/K) + (r + (sigma**2/2))*t)/(sigma*np.sqrt(t))
    d2 = d1 - (sigma * np.sqrt(t))
    C = S0*norm.cdf(d1) - K*np.exp(-r*t)*norm.cdf(d2)
    return C

Create relevant constants

In [27]:
#Risk free rate:
r = 0.035

#Set an initial portfolio cash value:
initial_cash = 10000
cash = initial_cash

#How many positions of the stock are held:
position = 0

#Portfolio value:
portfolio_value = []

#Trade log to see all buy and sell positions
trade_log = []

#Buy and sell signals
buy_signals = []
sell_signals = []

#Help calculate P&L
entry_price = None
pl_points = []

We can then download European option price data, I also include S&P 500 data to compare returns

In [28]:
data = yf.download("^STOXX50E", start="2024-01-01", end=datetime.today(), auto_adjust=False)
dataSPX = yf.download("^GSPC", start="2024-01-01", end=datetime.today(), auto_adjust=False)

#Calculate S&P 500 portfolio data
dataSPX['Value'] = (dataSPX['Close'] / dataSPX['Close'].iloc[0]) * initial_cash

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Compute our constants for the Black-Scholes model:

In [29]:
#Calculate returns
returns = np.log(data['Close']/data['Close'].shift(1)).dropna()

#Annualized Volatility
sigma_real = returns.std() * np.sqrt(252)

#Latest price
S0_real = data['Close'].iloc[-1]

#Strike Price
K = round(S0_real)

#Time to maturity
expiry = datetime(2025, 12, 19)
today = datetime.today()
T_real = (expiry - today).days / 365

Compute the Black-Scholes delta equation which represents the sensitivity of an option and follows:

$$\Delta = N(d_1)$$

Where 'N' is the cumulative distribution function of the normal distribution

In [30]:
def BS_delta(S, K, r, sigma, T):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    return norm.cdf(d1)

From this we define our buy and sell indicators using the Black-Scholes delta, we use a for loop here to scan over all the data:

In [31]:
for i in range(len(data)):
    # Scan data
    S = float(data['Close'].iloc[i])
    T_i = (data.index[-1] - data.index[i]).days / 365

    # If time to maturity has not reached 0
    if T_i <= 0:
        continue
    
    # Find delta
    delta = BS_delta(S, K, 0.035, sigma_real, T_i)

    # Check if we hold any positions and the sensitivity of the option
    if delta > 0.4 and position > 0:
        # Update our cash
        cash += position * S
        sell_signals.append((data.index[i], S))  
        # Find P&L
        PL = (S - entry_price) * position
        pl_points.append((data.index[i], S, PL))
        trade_log.append((data.index[i], 'SELL', S, position))
        # Reset position as we have sold them all
        position = 0
    
    # Check if we don't hold any positions so we can make a call
    elif delta < 0.4 and position == 0:
        position = cash // S
        entry_price = S
        # Update cash
        cash -= position * S
        buy_signals.append((data.index[i], S))  
        trade_log.append((data.index[i], 'BUY', S, position))

    # Portfolio data
    total_value = cash + position * S
    portfolio_value.append((data.index[i], total_value))

# Create dataframes after loop
pv_df = pd.DataFrame(portfolio_value, columns=['Date', 'Value']).set_index('Date')
comparison_df = pv_df.join(dataSPX['Value'], how='inner', lsuffix='_Portfolio', rsuffix='_SP500')

Then we simply plot our data

In [32]:
# Setup subplot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
fig.subplots_adjust(hspace=0.4)

ax1.plot(data.index, data['Close'])

# Annotate our buy and sell signals so we can visualize
if buy_signals:
    buy_dates, buy_prices = zip(*buy_signals)
    ax1.scatter(buy_dates, buy_prices, marker='^', color='green', label='Buy Signal')

if sell_signals:
    sell_dates, sell_prices = zip(*sell_signals)
    ax1.scatter(sell_dates, sell_prices, marker='v', color='red', label='Sell Signal')

# Annotate our profit and loss on the graph
for date, price, pl in pl_points:
    color = 'green' if pl >= 0 else 'red'
    ax1.annotate(f"{pl:.0f}", xy=(date, price), xytext=(0, 10), textcoords='offset points', ha='center', color=color, fontsize=8)

# Titles
ax1.set_title("Historical price of Euro Stoxx 50")
ax1.set_xlabel("Date")
ax1.set_ylabel("Price")

# S&P 500 plot against our portfolio increase
ax2.plot(comparison_df.index, comparison_df['Value_Portfolio'], label='My Portfolio', color='purple', linewidth=2)
ax2.plot(comparison_df.index, comparison_df['Value_SP500'], label='S&P 500 (Buy & Hold)', color='orange', linestyle='--')

ax2.axhline(initial_cash, color='gray', linestyle='--', label='Starting Capital')
ax2.set_title("Portfolio Value Over Time")
ax2.set_xlabel("Date")
ax2.set_ylabel("Value (Euros)")

ax1.legend()
ax1.grid(True)
ax2.legend()
ax2.grid(True)
plt.show()

Gives us a comparison on our strategy with S&P 500 returns over one year. The strategy uses delta thresholds to generate buy (delta < 0.4) and sell (delta > 0.4) signals. This provides a systematic approach to portfolio management based on options pricing theory.